# The REINFORCE loss with a learned baseline

Write the two losses of REINFORCE with a learned baseline (u20-l07) for one recorded episode of length $T$.

```python
def reinforce_losses(logits, actions, rewards, baseline, gamma):
    ...
    return policy_loss, value_loss
```

- `logits`: float tensor `(T, A)`, the policy network's action preferences at each step (it requires grad).
- `actions`: long tensor `(T,)`, the action taken at each step.
- `rewards`: float tensor `(T,)`; `rewards[t]` is $R_{t+1}$, the reward that followed `actions[t]`. The episode ends after step $T - 1$.
- `baseline`: float tensor `(T,)`, a value network's estimates $b_t = \hat v(S_t)$ (it requires grad too).
- `gamma`: the discount factor.

With the rewards-to-go $G_t = \sum_{k=t}^{T-1} \gamma^{k-t} R_{k+1}$ (compute them backwards, $G_t = R_{t+1} + \gamma G_{t+1}$) and $\log\pi(a_t \mid s_t)$ the log-softmax of `logits[t]` at `actions[t]`, return two 0-dimensional tensors:

$$\texttt{policy\_loss} = -\sum_{t=0}^{T-1} \log\pi(a_t \mid s_t)\,\big(G_t - b_t\big), \qquad \texttt{value\_loss} = \frac1T\sum_{t=0}^{T-1}\big(b_t - G_t\big)^2.$$

Gradients must flow as in a real training step:

- `policy_loss.backward()` gives gradient to `logits` only. The advantage $G_t - b_t$ is a constant for the policy, so the baseline enters it **detached**.
- `value_loss.backward()` gives gradient to `baseline` only ($G_t$ is a constant target).

Do not modify the inputs.

**Example.** With `gamma = 0.9`,

```python
logits = torch.tensor([[0.0, 0.0], [1.0, -1.0], [0.5, 0.5]], dtype=torch.float64, requires_grad=True)
actions = torch.tensor([0, 1, 1])
rewards = torch.tensor([1.0, 0.0, 2.0], dtype=torch.float64)
baseline = torch.tensor([1.5, 1.0, 1.0], dtype=torch.float64, requires_grad=True)
```

the rewards-to-go are $G = (2.62, 1.8, 2.0)$, the advantages $(1.12, 0.8, 1.0)$, the chosen log-probabilities about $(-0.6931, -2.1269, -0.6931)$, and the function returns `policy_loss` about `3.1710` and `value_loss` `0.9648`.

**Restrictions.** Only `torch` (with `torch.nn.functional` and `torch.distributions`).

**Constraints.** $T \le 1000$, $A \le 18$. Time limit 0.5 seconds for $T = 1000$.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem7_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem7_submission.py
import torch
import torch.nn.functional as F


def reinforce_losses(logits, actions, rewards, baseline, gamma):
    # Your code
    return policy_loss, value_loss


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem7_submission as submission
importlib.reload(submission)
# e.g. submission.reinforce_losses(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z2zaW/1dw7syG9MqM7LTdq1L3zk3dbHbyNYl3dm8sDwuJkIw1RbIEFdt1/b/f7z0AJEjJdpzuXTsTUyDw8L6/AN7s6GxnInbWB+O9avyXnZHYWalGYuhm802jm1zR2Mm5Eh+OX739+d2HF8ciL40Rl7o5F1LkStaFysRMGpXrQtGyXBlTFh5W7nbRxbpRBqP73+BnphcLPV/nzTVGnmFAFU1Nzzu10sWirOcqpX2wghaX2TpXaSFXjE5Vl7Ncrf6SmvVspY3R2A2zZJ6XlypL9aoq64a2Ot1pAOl85wxvZ7IAojTofgF+WdGshcyNInKVadJGr1S5blJ6cTCmUbm0oOo8qcpcz6/TZS0zDYRpU4y2GLvfIS94/yQDi4FG+zsvl6kpF81KXjFuplzTcpBW1nqpC5nT3ErWzQEGm3qtbvF7oXNm4M3OOTHdmKRijk35/7+qvFK1Ec25bITjkCBMadCyRES/LOpyJdxyP7r7S5xMi2lx5KYLbcS8LBpQ+HRd6MY8/Z7+/PDUQcXvSmc/PLWzgUQiXjUiUwvQjP3BRbFYF/MGYjGCZJbxYLprcWvkhcKTmhad+ISVsPgFQ78k4vifR2/evz7+KHJtGiJJWbAEzAhzXl4WAos+rAsRfdJGA6v4ufhI4BpRr7Gt+qTqa14E0n6WOl/XWFlLbZSgn15/Ac/IpRKXtW5AsIAcxY+qmAgjr8UloXspoex1WSxHoiCoIluvKnGuswzTdVGtG0Pcs1IoHEvnZXXd/iCVIv66nwXWXwtALSoa/Uq8KHM5Ey/e/90IWSthSItr4lVBqDwx4s2BeF+XI2FK2N5c5gRRF0sjVmswZaaIqWBRSSal8TrXxIcZ6D+HKBdyDq1LaKeDZEzClWK5JvlHdmcDWLnaa85rJTNhKgWBYVZdrpfn+TW0JV8I4AIs5mVNjK4VMb0RekEI4ifbm6PDGRAx5eP74+Of0p+PXpy8+yAOaXcieFrMc4ndSQzR8dVcVaQp8WRaCPxX4ZWdBYUCx1UWQRezkViZpZ+DfYuSmFxkboT+64Qb0dwOSFpU0ZVfCy8TrPESIqsMRhegXxemkcVcRVcj+z45UYUp6zhYztuqZl0X4soZeRQn82qNf1nMUWwnKyZTvOLtjuu6rAMoluQAVlEl0si6ltfAu6PDnMtKRcuyGUEpC/oX+nk4hcc0cKTTHY+ZAa+bdZWriCjH/DjhpQ4X5inm+EkEKx6JxXTnhgDeQuDG7iVuDPyOuqrUvIFO3ATzb7Fdh9kcrvo+zEaibsr8cF/tfTsS0j3+p8cXlnUJlFtkpWmuQegiLyUhRuO8Z/9FQMzS0kckXdrHO8ixBLjp8Tba/Csmr90AAtEG7k03WA088jyKBTwFaWH47tK9C7cnTyo1e8KC1iBWtMCdIpO889wxEcxw7KJ/HLvon1DxVF2DY8wIQg5xhP7ImYmWYk9cxnFMsEGo/k0JhfAmxmR8W0ylzyd2dOKTzOEfRJTLekluV5HCihv8mSTPlrdxT/bq17XMP0Mr+0ZlpxLSRcaaPoqZo0BCNk3t3k93WBwdkC3qEihIN+f/XC1C1WD8U8cHILdhTCFXWwgklckQYygDYcvEdzDgjG/w6j/qHmo0C0M9YWiTsomwOOjJ3CkP3jCQSLdkA38aBSsc4DhJORNL0x46S7g/kpzb9Ymf9MRD7WOamkJWiOTN/69rzstCPdIpY0uvlVekkhzbd2KExIytt49PlOk5GEmJSxzi0yeuhx0BjDKlqsMTZHqBgjkET8C+IXqbAOJeCKGhhGDyu6se4yGWSI7E7Mv5Lj+L73aKtQve70vZLontDMf0XYHbSNpNTDSLt/kaLO/8zNblrZ+YuUeWbd+yiYKR9XbQ7CKUlYfDEILkpUxXaxgFsptoUYzELtwpLGN39+KSnjwmNnV8gQggFgXvvKAElWhAjoMscAnzkgVSxnq5XiEpRwgRVS7nKvFZJ0d9sif4xdPOsmTM+aykBbTjmZ2J7IyCRxENEbKvaYmGIoNgE9NShVRG1RLx7TddRXYN79FTcPYnrXI5J3KNwga6mClK8PVCU2ZJKXNHy1c3WvxZ7N8+F5dlfUFZvWTlFSQ+5KOtv3RMBvIdhymJ10VkFKWCwOpzufw8ZDG8a5hyb8mkubBEPDTIvUEBMYhz3SftSiTvtoaird4BRE0lGAqklSGSOA9/I+dijg1oCYi0WTdVK5zYU3KA6mJvts7gSWmYc3Ck5ZoKqHUFcxLNpZ4rAggOaUTocuEsGDh7jhpWIRpZIF+i8E0QsGKNd4mgUn6hawyDmPwJbUFxwW5GHFZZT62aMWWTgJCgulykDEbV3ts8oEpZs30xcpTGpSOO3kPhhCieirBs6FQyJVWsyRqig1DvIERs8/2hgzRwRTPUNBfdkDwYiYsD7Nb3j4zyaDDYo+RhVjCexIgDZsNBMMxsQMUWZRRi72BHmG+GBLEdNWXJeoPiP2smycHiFlk+tEw9d1VXq8SsTDA7BiTWBdXzN441bh3XkCjsvIi3/MdborJEGbyAai0pgbVF5o2TEwMi67jXQOtiCetUmRdYV+BAklm5ArcXEuqXthOn3MpZGyl1yY0W1etyvKQRWAAw65oHmp3Guc4zsro5dUZQMGuyBZr5t4/v3gpqyJDlmCYDhmyr1XVzjpG9V8LvI753LY400/UP4vtuD7zEb1jl7zzW/LBR8sPYgvIfgr1qcj1rh+wfjCTrRufdcNk+/suUXZOgks15uNzoZSG7ZebaDNoL/keNuDCT84t2BFpcULPA9hrMXIPMQrXuDgMXutnjPp7YT75N9l1OZybUJ1pRM2/ELgXVhQs+3vlVUoPLy2nh90gWOodG+5/RdAdol7WiCtB1WpCJJrt/Ly4K6uGYMqeOSskdAGyoK5Jasusyxb8efziG5TheJO/xN0pTaoOlaZzAcdFyZHWVrIHZtABXEpqcQFtV3URjGAPSBwITA+C23hf3gsRXiNW/yok4/np8wJ2w1/84+p+P6Y9Hb98e/wQMbqY7qWsrpikRA8K8PO1PdaXm7gl5vn2al6sKuE53bqfF63fv3qdv3/10/BHgIuhK8nMJW6WHf0BxlX08MtfFvH3xGjnkC8Cwvz6q4MdPSDK7Xy9VQfG5rI+vqjhssZzYXsxDXZZUotJbRaRj6xX8DcKWai2WS0UPKOxGUGYzT+fnan5hIlPPScSN7MfbD9beKRtGsCI+lheWOSuzBGOewy1SBmh7fGw2pnnSSoeTIddKpEDV1Nq2FXshygah07Nt2WxTKyR2zCVoCeprINrPQz9eoz6/4jyUmnJqMz88dWhPoC3cKXbYT8hNGl7uSuSysH7mRiX0tyhvJ/SM2Sh+bh2Crk1NIQH8SuCYYSeD3jXx6PTMITqT7TjHycaHHG6HpmvDwILxGewhtb1SN04qjpQRtqszmAplYrVNFCxgQLDzI6poKOnWM2rXg5wLzt1UnpBiuw24cx7wfCWb1ToPRxbcHYGrJt8M5l/K/CIiYQxCd5Cm03Srz7Y0GFYVbSpLExNGd7IZwFxLBflyJHkSJeUrai3Ynwl2AyMpc4xWyDzhjqiYI+ArRtYKIp5sj42BLBKZZW6PeHNyJ5tgXmKqXJO4ac/Tvf2zLQs74SXrKqOse8vasEjM72fjz/B6Q2ogUMiK+ega7yC/M6gtvOQVjpF4/gNcHHIQ0AZcCFjX48A2DRgsvYt5X7SU6Xws04+87QxpHygE40GG9llQ3wYe+T6AOrubqu2Ttu3YBavhnmz4iawqVWSR7/jYRdMd6/Kse/+vPr+27cJR8EddvHNR7Gi9PDIUg2LbWxksSEo3741s3iBl3NBp64MeiRwQCxQyAElJogN1p/v3WYC1EqvyNMoGQfOSf5VUnrKDjYJ94ni7d/kzRRQwxoO6eTISTywQb1FkgE+wH+re5RNqxd5u7yduw59aFlvQNx73DthX7QniTBaGeDs/dxGCU8EuQHD7WUkUsf/kv/FzW43ml/La7NlDV1QyyAt1C4iU0kEri/x65PdE5dKFmshqgW0iuVSKko5rkaslSvNLSpsEz0ySJA6CGZIykZUkFAv4XHN9aaUQBUHzTxwcgyhs0XXBF2r4u4iCWPon0csLAxXCDo9WHWBghOMPwY9MvEV3ADn+IxIuSr9HeyTbSRmY94lPrRGlZWXzyX63jaZbIxuY3ucQTBUl9OK/hYdNJWtgNyKymZPb4XR8Fir3FvIfw4Ju1/uIZ+e2QTKPfhnBNk1iZe8RimEyY1LqWp2rgovYNnfkVf9e+h1lt/dU6otaA0x+nXImGwEECt2Uqql+Rv+uUHuMZ6bMvNZcVYgKqgr7Xgp3Es0TtD++poIN5qponuHD+n6HaUZtFV+3Jqib8dykzSxSSZq2L1DyuWSTgCPTXHBQX9A+AAFJLaj8VD7no8qvJcFlpZfUMqGGFXlZ7k06pcO/lI4FWXtviETBCkOz+TitQ98xk05M+IREBecjlP3f3vCutzTfM3uFTDxqazF42mGVS5WsrJefTtsU0ZNy58yDs6AkdpyiBPxQtFOe+XQdeo9xajgkeSkzE0WMxFNSG7IJbkXsEDzELOpkRN7ZoXwiiA6ZcILnBncTD8WmXdiCcepu4Ez5uo3pftrGjFVZV0mTJtLA2xIiCOGfdtDOCJ07StHW0CmDJRM0pxarM44W9Iut5CxMZ31J5/NZw33HjW1DL8Fdi4jZSZdDTGSnh1HeaolveS6AtC6Tjw21T16988yzTZ7E/oncr4+vXh69/vBm5Kp0NxXuLOXmrj8FDnxZ/x4T8fJg7HHpV8d8CaZrVkGcma4VjM8KIwKmwzTL46gaTVvUHs1XJ6/eHH9IPxwfvR612A3SHFMp0p5+Kyyh0ZR6M7azQ91JPjIJKAruf1nV6BpzEFPgqgb7rWeb2zlYvCFtHdE/w4WwGDvPnNJrLhjOrOJvksQ2pOqEekCpXUaeJ/4Cxo37vQnXdum1JJwiOuM4s97M0kgemL0ae9ymLC/4TEHceHlMkjF1gL3rcrv8KI1qO0S2B9K1xH58fTwe73cIPIqMuzB2B0OMJp3E0A2x0gnKJUH3xKPWrDdAb2OU9yu0L/SZNIpP4iMU4F+Px+PJ2RcbMt9Z7PdxeCi832Hn+LNif6IJWkb2VfywcDteZaWySZO9bIfoQiBuw7r9kSQsH2eT0x176Y+ssA0Z7T3Alm53j/Bh21sGxre8y5YsNDfLXfdzjS1f5dkp5Bn81UGfxDu3R7GGkoYIhcrCOnZ6It/+SdbG75HoRq1MxLdnin5UYK/qz/3pAEUCjWgRhw0veAk6hOSUhHYcJM8cjkn76zX8llUKylb4FkHhSRs4XHLPulirUH5zG2CtP5wwkNNvJmfw33wuDGxdl1qwe5zuONB+crAbvaVuMJU928oU52/toQ8NjZPx7WPOxDYvPnxB1Pmjkccdzm165S9yaU4Ipx3nyFIp8d68u7HhwbvFNvewNs7OOi+LpT+XGfpsEenC3jLjbH5EBcRC1sKfDMahE3Cb8+HGoKf9Bwlucab0estdlUfFkn8XLuE9Az6vyD4zhnSwWgUneDW0OIvuOqgdiWfbwptLZ898QVa3ju3RoahdYHPVQyodoy53vS8rZacyK8s82kDMviNQTU9zCVzTA9eu8AXKnSGFrzUvhC94rHdLU6pv6KBs4o8HqNyxJ7ttsKAz3f5do7vO50Z8lj2yt0xH4bWakbsBQmi0d8bh5Z23VldyVZHXA1/pGM/2kjhf/vbr4JCa46FJmzJdliDuUtZ0mWQpV6v2OOsEy1Fb+reOMy9pMzjEM7ErTro40HRXFU5CT/bytLFJ5Cpi4GJ3V0QXpFQxANg83m1wenFmo9RFB6sZiZO4X8Jbihq+hRW9HImMStDDn+KQuAWqT2qi5uVSN3SWa9tAI9FS6j9YGNBM1N3LG3cYVBFvGDiR4j9wAJXqqmp3zfTqcH8kLpSq6DG4PZVXqWyFA02eX0SneXXatJiCa2dDvpIs3Os4bhW1zAFoL2KIYO1L4ONpa+8AxnFCAvDphKQlUeSnYcXLmORyEHM7c3CtrqIbuXTA211/rbSB3+HLC5DPSBwdPmsvNrdktSe0EcDKgu6QGb6Gba882NY6c9CvoPsRRUTwwG2/+nDZiRhh89c14qbhWxKpR9QxpQeHzHfMkAAw7sFryWPBbu6+dWt/yuhY9nmL7sDXMfZh5Qxum7M7pAzaXjpiZRwJJGSqvSRtz4TDG3i5XM0yqMWVmVAYSIbfGUV44xXb3SKLt9+UBeyRvTtvXSrpIsZi8n4H9u4s43I76W4G8nQRuQ+IaMeRvRDMz3GbOXsFA/LcnuunlldkABHSuwAOp+JlDpwx3oGkYTqw3biid8/1WabmKoF9RkzMuE/MDSFx6z86AVHjPUy17Uv6KIWBDK8DbVoMt2lSexv6ITEG7LhX6q69J4sm5SX8ZNd1DnA3WBj0hmbGflEQ+eVxLH4Q9K1CcADR3s/H+9aVjMSeXxOM1cPvG7bcuPcsDeRob1qfK3dbnNKgCd+MWenf6LB+heSPHvmCl2MufVRn7rrHRZcT1qu0IdsSlY4kHn8XcOUx+0b8gnfEr1ZiLYGjgJN3YBt+0/Fd903H/rgHCxLoZNGH1dfUu0C1XtZG8rZnOnCWLgCeUiweUYVyhuJvn5739u2PcfINvfjm7Kz1SORjAo8UhqShE/XwARAxrO3JDr2mn7bvsBjRJ09nD/rNbhlw5LX74bJ78PwC5xkmRlwPOY5uWmXLc6LlO3vJyeVT8QCe/RoudbHQhHA5h+HA4wyvf6P4NNpn8N+R//oLP9LTgWUCPe4fMC/jXsd1C7YuDu+PkfhfUKbUugionftcz00SNxd85xhYUSF9cuum0g9+GHziwjQOvv0MaXyY/4DsESTSvqM04evQ5ac9D/cZuZqVyjcj138hlBjP/odL5Jl62sPlfuB0go4SVS18CmVtqznnDwCFXDd0kTkTkT1TFdbT2XOecrlHbaAggC1Tj/syJVxbLfdwEvrHepmWzIAoPiVL1wUd0Iaqbr+PsSC14SMBLoHZc9tx/v4p5o+hYhu/YEuBa7QnkJZ6+xGzUXTjuOUf8qSS6fL4TATQ2Ewhub9Dh9zZJ/g2+ny09aYtH8jr3UH741JyknJ8Sv7AzqbnHkucvNxxKjFmKOSl/sRS7ihtSgdt4Pw76fkPsNol5aIXrbiJA62iq6IhuCD+Bb78YGhN1v3/YWN6BmP61lGQbmYKn21HD5gR4G6aURfBtjM4VKRtxtO+o88FVGhB9GbUWtIdWsQRNer0xGnHgxbUqcvQhuybrVbUJ3eb2QzU6Z7akVx8X+csvQeUlmzaGhdkTznJfhkPNTJAaqiQIYuhlmGK9FkaSl9PhHpJyWOgeV9ThBqPqaw69BroPkfhXGNbidF+nMIBb1rs3N7+L9OCRmE=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
